# SALES METRICS ANALYSIS

## WEEKLY TABLE

###  Import Libraries and Set Up

In [1]:
# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
from datetime import datetime
import os
from openpyxl import load_workbook
from openpyxl.styles import Font

# Display settings for better output viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries imported successfully!")

Libraries imported successfully!


### Load Reference Dates

In [2]:
# Step 2: Load the analysis reference date file
# This file contains the date ranges we need to analyze
# NOTE: analysis_ref_date.csv is in the root directory (same level as data/, outputs/ folders)
# Since this notebook is in the 'notebooks' folder, we need to go up one level with '../'

# The CSV has separator lines (===) and multiple sections, so we need to handle it carefully
ref_date_path = '../analysis_ref_date.csv'

# Read the file and skip rows that are just separators
# We'll read it line by line to handle the special format
with open(ref_date_path, 'r') as f:
    lines = f.readlines()

# Find the REPORT section and extract Week Start Date and Week End Date
week_start_date = None
week_end_date = None

for line in lines:
    # Look for lines containing "Week Start Date" in the REPORT section
    if 'Week Start Date' in line and week_start_date is None:
        # Extract the date after the comma
        parts = line.strip().split(',')
        if len(parts) >= 2:
            week_start_date = parts[1].strip()
    
    # Look for lines containing "Week End Date" in the REPORT section
    if 'Week End Date' in line and week_end_date is None:
        # Extract the date after the comma
        parts = line.strip().split(',')
        if len(parts) >= 2:
            week_end_date = parts[1].strip()
    
    # Break once we have both dates from REPORT section
    if week_start_date and week_end_date:
        break

# Display the extracted dates
print("Reference Dates Loaded:")
print(f"Week Start Date: {week_start_date}")
print(f"Week End Date: {week_end_date}")

# Create the "This Week" date range string (format: MM/DD/YYYY - MM/DD/YYYY)
this_week_range = f"{week_start_date} - {week_end_date}"

print(f"\nThis Week Range: {this_week_range}")

# Calculate "Previous Week" range
# Parse the dates to calculate previous week
this_week_start = pd.to_datetime(week_start_date, format='%m/%d/%Y')
this_week_end = pd.to_datetime(week_end_date, format='%m/%d/%Y')

# Previous week is 7 days before this week
prev_week_start = this_week_start - pd.Timedelta(days=7)
prev_week_end = this_week_end - pd.Timedelta(days=7)

# Format back to MM/DD/YYYY - MM/DD/YYYY
prev_week_range = f"{prev_week_start.strftime('%m/%d/%Y')} - {prev_week_end.strftime('%m/%d/%Y')}"

print(f"Previous Week Range: {prev_week_range}")

Reference Dates Loaded:
Week Start Date: 11/30/2025
Week End Date: 12/06/2025

This Week Range: 11/30/2025 - 12/06/2025
Previous Week Range: 11/23/2025 - 11/29/2025


### Define Helper Functions

In [3]:
# Step 3: Define helper functions for data extraction and calculation

def get_value_from_csv(csv_path, date_range, column_name):
    """
    Extract a value from a CSV file based on date range and column name
    
    Parameters:
    - csv_path: Path to the CSV file
    - date_range: Date range string to match (e.g., "11/30/2025 - 12/06/2025")
    - column_name: Name of the column to extract value from
    
    Returns:
    - The value if found, otherwise returns blank (None or empty string)
    """
    try:
        # Read the CSV file
        df = pd.read_csv(csv_path)
        
        # Check if required columns exist
        if 'Date Range' not in df.columns or column_name not in df.columns:
            print(f"Warning: Required columns not found in {csv_path}")
            return ''
        
        # Filter by date range
        row = df[df['Date Range'] == date_range]
        
        # If row exists, get the value
        if not row.empty:
            value = row.iloc[0][column_name]
            # Check if value is NaN or blank
            if pd.isna(value) or value == '':
                return ''
            return value
        else:
            # Date range not found
            return ''
    
    except Exception as e:
        print(f"Error reading {csv_path}: {str(e)}")
        return ''


def get_value_from_excel(excel_path, sheet_name, date_range, row_name):
    """
    Extract a value from an Excel file based on date range (column) and row name
    
    IMPORTANT: Handles both 2-digit year (MM/DD/YY) and 4-digit year (MM/DD/YYYY) formats
    
    Parameters:
    - excel_path: Path to the Excel file
    - sheet_name: Name of the sheet to read
    - date_range: Date range string to match in column headers (4-digit year format)
    - row_name: Name of the row to extract value from
    
    Returns:
    - The value if found, otherwise returns blank
    """
    try:
        # Read the Excel file
        df = pd.read_excel(excel_path, sheet_name=sheet_name)
        
        # Convert 4-digit year date range to 2-digit year format
        # Example: "11/23/2025 - 11/29/2025" becomes "11/23/25 - 11/29/25"
        date_range_2digit = date_range.replace('/2025', '/25').replace('/2024', '/24').replace('/2026', '/26')
        
        # Check if either format exists as a column
        date_column = None
        if date_range in df.columns:
            date_column = date_range
        elif date_range_2digit in df.columns:
            date_column = date_range_2digit
        else:
            print(f"Warning: Date range '{date_range}' (or '{date_range_2digit}') not found in {excel_path}")
            return ''
        
        # Set the first column as index
        df = df.set_index(df.columns[0])
        
        # Check if row_name exists
        if row_name not in df.index:
            print(f"Warning: Row '{row_name}' not found in {excel_path}")
            return ''
        
        # Get the value
        # Note: If there are duplicate row names, .loc returns a Series
        # We need to handle this case
        value = df.loc[row_name, date_column]
        
        # If value is a Series (multiple rows with same name), take the first one
        if isinstance(value, pd.Series):
            value = value.iloc[0]
        
        # Check if value is NaN or blank
        if pd.isna(value) or value == '':
            return ''
        
        return value
    
    except Exception as e:
        print(f"Error reading {excel_path}: {str(e)}")
        return ''


def calculate_wow_change(current_value, previous_value):
    """
    Calculate Week-over-Week percentage change
    
    Formula: (Current - Previous) / Previous * 100
    
    Parameters:
    - current_value: Value for this week
    - previous_value: Value for previous week
    
    Returns:
    - Formatted string with + or - sign and % symbol
    - Returns blank if calculation cannot be performed
    """
    try:
        # Check if either value is blank/empty
        if current_value == '' or previous_value == '' or current_value is None or previous_value is None:
            return ''
        
        # Convert to float (handle percentage strings like "3.58%")
        if isinstance(current_value, str):
            current_value = current_value.replace('%', '').strip()
            if current_value == '':
                return ''
            current_value = float(current_value)
        
        if isinstance(previous_value, str):
            previous_value = previous_value.replace('%', '').strip()
            if previous_value == '':
                return ''
            previous_value = float(previous_value)
        
        # Avoid division by zero
        if previous_value == 0:
            return ''
        
        # Calculate percentage change
        wow_change = ((current_value - previous_value) / previous_value) * 100
        
        # Format with + or - sign
        if wow_change >= 0:
            return f"+{wow_change:.2f}%"
        else:
            return f"{wow_change:.2f}%"
    
    except Exception as e:
        print(f"Error calculating WoW change: {str(e)}")
        return ''


def safe_sum(*values):
    """
    Safely sum values, ignoring blanks/empty strings
    
    Parameters:
    - *values: Variable number of values to sum
    
    Returns:
    - Sum of non-blank values, or blank if all values are blank
    """
    try:
        # Filter out blank values
        valid_values = []
        for v in values:
            if v != '' and v is not None and not pd.isna(v):
                # Convert to float if string
                if isinstance(v, str):
                    v = v.replace('%', '').strip()
                    if v != '':
                        valid_values.append(float(v))
                else:
                    valid_values.append(float(v))
        
        # If no valid values, return blank
        if len(valid_values) == 0:
            return ''
        
        # Return sum
        return sum(valid_values)
    
    except Exception as e:
        print(f"Error in safe_sum: {str(e)}")
        return ''


def safe_divide(numerator, denominator):
    """
    Safely divide two values, handling blanks and zero division
    
    Parameters:
    - numerator: Top value
    - denominator: Bottom value
    
    Returns:
    - Result as percentage string, or blank if cannot calculate
    """
    try:
        # Check if either value is blank
        if numerator == '' or denominator == '' or numerator is None or denominator is None:
            return ''
        
        # Convert to float
        if isinstance(numerator, str):
            numerator = float(numerator.replace('%', '').strip())
        if isinstance(denominator, str):
            denominator = float(denominator.replace('%', '').strip())
        
        # Avoid division by zero
        if denominator == 0:
            return ''
        
        # Calculate percentage
        result = (numerator / denominator) * 100
        
        return f"{result:.2f}%"
    
    except Exception as e:
        print(f"Error in safe_divide: {str(e)}")
        return ''

print("Helper functions defined successfully!")

Helper functions defined successfully!


### Extract Data for Previous Week

In [4]:
# Step 4: Extract all metrics for Previous Week

print(f"\n{'='*60}")
print(f"EXTRACTING DATA FOR PREVIOUS WEEK: {prev_week_range}")
print(f"{'='*60}\n")

# Initialize dictionary to store previous week values
prev_week_data = {}

# 1. Visits
prev_week_data['Visits'] = get_value_from_csv(
    '../outputs/pro_site/weekly_pro_site.csv',
    prev_week_range,
    'Visits'
)
print(f"Visits: {prev_week_data['Visits']}")

# 2. CVR %
prev_week_data['CVR %'] = get_value_from_csv(
    '../outputs/pro_site/weekly_pro_site.csv',
    prev_week_range,
    'Conversion Rate'
)
print(f"CVR %: {prev_week_data['CVR %']}")

# 3. Partial Typeform Submissions
prev_week_data['Partial Typeform Submissions'] = get_value_from_csv(
    '../outputs/typeform_submissions/weekly_typeform_submissions.csv',
    prev_week_range,
    'Partials'
)
print(f"Partial Typeform Submissions: {prev_week_data['Partial Typeform Submissions']}")

# 4. Complete Typeform Submissions
prev_week_data['Complete Typeform Submissions'] = get_value_from_csv(
    '../outputs/typeform_submissions/weekly_typeform_submissions.csv',
    prev_week_range,
    'Completed'
)
print(f"Complete Typeform Submissions: {prev_week_data['Complete Typeform Submissions']}")

# 5. Booked Calls (Closers)
prev_week_data['Booked Calls (Closers)'] = get_value_from_csv(
    '../outputs/discovery_intro/weekly_discovery_call.csv',
    prev_week_range,
    'Booked Calls (Completed)'
)
print(f"Booked Calls (Closers): {prev_week_data['Booked Calls (Closers)']}")

# 6. Follow-up Calls
prev_week_data['Follow-up Calls'] = get_value_from_excel(
    '../data/stat.xlsx',
    'stat',
    prev_week_range,
    'Follow ups'
)
print(f"Follow-up Calls: {prev_week_data['Follow-up Calls']}")

# 7. S2C Calls
prev_week_data['S2C Calls'] = get_value_from_excel(
    '../data/stat.xlsx',
    'stat',
    prev_week_range,
    'S2C'
)
print(f"S2C Calls: {prev_week_data['S2C Calls']}")

# 8. Total Closer Calls Booked (calculated)
prev_week_data['Total Closer Calls Booked'] = safe_sum(
    prev_week_data['Booked Calls (Closers)'],
    prev_week_data['Follow-up Calls'],
    prev_week_data['S2C Calls']
)
print(f"Total Closer Calls Booked: {prev_week_data['Total Closer Calls Booked']}")

# 9. Closer Availability
prev_week_data['Closer Availability'] = get_value_from_excel(
    '../data/stat.xlsx',
    'stat',
    prev_week_range,
    'Capacity'
)
print(f"Closer Availability: {prev_week_data['Closer Availability']}")

# 10. Closer Fill Rate (calculated)
prev_week_data['Closer Fill Rate'] = safe_divide(
    prev_week_data['Total Closer Calls Booked'],
    prev_week_data['Closer Availability']
)
print(f"Closer Fill Rate: {prev_week_data['Closer Fill Rate']}")

# 11. Booked Calls (Setters)
prev_week_data['Booked Calls (Setters)'] = get_value_from_csv(
    '../outputs/discovery_intro/weekly_intro_call.csv',
    prev_week_range,
    'Booked Calls (Completed)'
)
print(f"Booked Calls (Setters): {prev_week_data['Booked Calls (Setters)']}")

# 12. Scheduled Calls
prev_week_data['Scheduled Calls'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    prev_week_range,
    'Sched. calls'
)
print(f"Scheduled Calls: {prev_week_data['Scheduled Calls']}")

# 13. Live Calls
prev_week_data['Live Calls'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    prev_week_range,
    'Live calls'
)
print(f"Live Calls: {prev_week_data['Live Calls']}")

# 14. Show Rate
prev_week_data['Show Rate'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    prev_week_range,
    'Show %'
)
print(f"Show Rate: {prev_week_data['Show Rate']}")

# 15. Average Days Booked Out
prev_week_data['Avg. Days Booked Out'] = get_value_from_csv(
    '../outputs/discovery_intro/weekly_discovery_intro_blended.csv',
    prev_week_range,
    'Avg Lead Time (Days)(Completed)'
)
print(f"Avg. Days Booked Out: {prev_week_data['Avg. Days Booked Out']}")

# 16. Offers
prev_week_data['Offers'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    prev_week_range,
    'Offers'
)
print(f"Offers: {prev_week_data['Offers']}")

# 17. Offer Rate
prev_week_data['Offer Rate'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    prev_week_range,
    'Offer %'
)
print(f"Offer Rate: {prev_week_data['Offer Rate']}")

# 18. Closes
prev_week_data['Closes'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    prev_week_range,
    'Close 1'
)
print(f"Closes: {prev_week_data['Closes']}")

# 19. Offer to Close
prev_week_data['Offer to Close'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    prev_week_range,
    'Offer to Close %'
)
print(f"Offer to Close: {prev_week_data['Offer to Close']}")

print("\nPrevious Week data extraction complete!")


EXTRACTING DATA FOR PREVIOUS WEEK: 11/23/2025 - 11/29/2025

Visits: 3294
CVR %: 3.94%
Partial Typeform Submissions: 
Complete Typeform Submissions: 
Booked Calls (Closers): 83
Follow-up Calls: 28.0
S2C Calls: 16.0
Total Closer Calls Booked: 127.0
Closer Availability: 115.0
Closer Fill Rate: 110.43%
Booked Calls (Setters): 40
Scheduled Calls: 118.0
Live Calls: 79.0
Show Rate: 64.25%
Avg. Days Booked Out: 4.04
Offers: 52.0
Offer Rate: 58.65%
Closes: 27.0
Offer to Close: 36.2%

Previous Week data extraction complete!


### Extract Data for This Week

In [5]:
# Step 5: Extract all metrics for This Week

print(f"\n{'='*60}")
print(f"EXTRACTING DATA FOR THIS WEEK: {this_week_range}")
print(f"{'='*60}\n")

# Initialize dictionary to store this week values
this_week_data = {}

# 1. Visits
this_week_data['Visits'] = get_value_from_csv(
    '../outputs/pro_site/weekly_pro_site.csv',
    this_week_range,
    'Visits'
)
print(f"Visits: {this_week_data['Visits']}")

# 2. CVR %
this_week_data['CVR %'] = get_value_from_csv(
    '../outputs/pro_site/weekly_pro_site.csv',
    this_week_range,
    'Conversion Rate'
)
print(f"CVR %: {this_week_data['CVR %']}")

# 3. Partial Typeform Submissions
this_week_data['Partial Typeform Submissions'] = get_value_from_csv(
    '../outputs/typeform_submissions/weekly_typeform_submissions.csv',
    this_week_range,
    'Partials'
)
print(f"Partial Typeform Submissions: {this_week_data['Partial Typeform Submissions']}")

# 4. Complete Typeform Submissions
this_week_data['Complete Typeform Submissions'] = get_value_from_csv(
    '../outputs/typeform_submissions/weekly_typeform_submissions.csv',
    this_week_range,
    'Completed'
)
print(f"Complete Typeform Submissions: {this_week_data['Complete Typeform Submissions']}")

# 5. Booked Calls (Closers)
this_week_data['Booked Calls (Closers)'] = get_value_from_csv(
    '../outputs/discovery_intro/weekly_discovery_call.csv',
    this_week_range,
    'Booked Calls (Completed)'
)
print(f"Booked Calls (Closers): {this_week_data['Booked Calls (Closers)']}")

# 6. Follow-up Calls
this_week_data['Follow-up Calls'] = get_value_from_excel(
    '../data/stat.xlsx',
    'stat',
    this_week_range,
    'Follow ups'
)
print(f"Follow-up Calls: {this_week_data['Follow-up Calls']}")

# 7. S2C Calls
this_week_data['S2C Calls'] = get_value_from_excel(
    '../data/stat.xlsx',
    'stat',
    this_week_range,
    'S2C'
)
print(f"S2C Calls: {this_week_data['S2C Calls']}")

# 8. Total Closer Calls Booked (calculated)
this_week_data['Total Closer Calls Booked'] = safe_sum(
    this_week_data['Booked Calls (Closers)'],
    this_week_data['Follow-up Calls'],
    this_week_data['S2C Calls']
)
print(f"Total Closer Calls Booked: {this_week_data['Total Closer Calls Booked']}")

# 9. Closer Availability
this_week_data['Closer Availability'] = get_value_from_excel(
    '../data/stat.xlsx',
    'stat',
    this_week_range,
    'Capacity'
)
print(f"Closer Availability: {this_week_data['Closer Availability']}")

# 10. Closer Fill Rate (calculated)
this_week_data['Closer Fill Rate'] = safe_divide(
    this_week_data['Total Closer Calls Booked'],
    this_week_data['Closer Availability']
)
print(f"Closer Fill Rate: {this_week_data['Closer Fill Rate']}")

# 11. Booked Calls (Setters)
this_week_data['Booked Calls (Setters)'] = get_value_from_csv(
    '../outputs/discovery_intro/weekly_intro_call.csv',
    this_week_range,
    'Booked Calls (Completed)'
)
print(f"Booked Calls (Setters): {this_week_data['Booked Calls (Setters)']}")

# 12. Scheduled Calls
this_week_data['Scheduled Calls'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    this_week_range,
    'Sched. calls'
)
print(f"Scheduled Calls: {this_week_data['Scheduled Calls']}")

# 13. Live Calls
this_week_data['Live Calls'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    this_week_range,
    'Live calls'
)
print(f"Live Calls: {this_week_data['Live Calls']}")

# 14. Show Rate
this_week_data['Show Rate'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    this_week_range,
    'Show %'
)
print(f"Show Rate: {this_week_data['Show Rate']}")

# 15. Average Days Booked Out
this_week_data['Avg. Days Booked Out'] = get_value_from_csv(
    '../outputs/discovery_intro/weekly_discovery_intro_blended.csv',
    this_week_range,
    'Avg Lead Time (Days)(Completed)'
)
print(f"Avg. Days Booked Out: {this_week_data['Avg. Days Booked Out']}")

# 16. Offers
this_week_data['Offers'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    this_week_range,
    'Offers'
)
print(f"Offers: {this_week_data['Offers']}")

# 17. Offer Rate
this_week_data['Offer Rate'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    this_week_range,
    'Offer %'
)
print(f"Offer Rate: {this_week_data['Offer Rate']}")

# 18. Closes
this_week_data['Closes'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    this_week_range,
    'Close 1'
)
print(f"Closes: {this_week_data['Closes']}")

# 19. Offer to Close
this_week_data['Offer to Close'] = get_value_from_csv(
    '../outputs/sales_tracker/weekly_sales_tracker.csv',
    this_week_range,
    'Offer to Close %'
)
print(f"Offer to Close: {this_week_data['Offer to Close']}")

print("\nThis Week data extraction complete!")


EXTRACTING DATA FOR THIS WEEK: 11/30/2025 - 12/06/2025

Visits: 4010
CVR %: 3.29%
Partial Typeform Submissions: 
Complete Typeform Submissions: 
Booked Calls (Closers): 131
Follow-up Calls: 25.0
S2C Calls: 30.0
Total Closer Calls Booked: 186.0
Closer Availability: 247.0
Closer Fill Rate: 75.30%
Booked Calls (Setters): 61
Scheduled Calls: 178.0
Live Calls: 124.0
Show Rate: 61.31%
Avg. Days Booked Out: 3.53
Offers: 77.0
Offer Rate: 56.27%
Closes: 33.0
Offer to Close: 23.83%

This Week data extraction complete!


### Calculate WoW Changes

In [6]:
# Step 6: Calculate Week-over-Week changes for all metrics

print(f"\n{'='*60}")
print("CALCULATING WEEK-OVER-WEEK CHANGES")
print(f"{'='*60}\n")

# Initialize dictionary to store WoW changes
wow_changes = {}

# Get the list of metrics (keys from either dictionary)
metrics = list(prev_week_data.keys())

# Calculate WoW for each metric
for metric in metrics:
    prev_value = prev_week_data[metric]
    this_value = this_week_data[metric]
    
    wow = calculate_wow_change(this_value, prev_value)
    wow_changes[metric] = wow
    
    print(f"{metric}: {wow}")

print("\nWoW calculations complete!")


CALCULATING WEEK-OVER-WEEK CHANGES

Visits: +21.74%
CVR %: -16.50%
Partial Typeform Submissions: 
Complete Typeform Submissions: 
Booked Calls (Closers): +57.83%
Follow-up Calls: -10.71%
S2C Calls: +87.50%
Total Closer Calls Booked: +46.46%
Closer Availability: +114.78%
Closer Fill Rate: -31.81%
Booked Calls (Setters): +52.50%
Scheduled Calls: +50.85%
Live Calls: +56.96%
Show Rate: -4.58%
Avg. Days Booked Out: -12.62%
Offers: +48.08%
Offer Rate: -4.06%
Closes: +22.22%
Offer to Close: -34.17%

WoW calculations complete!


### Create DataFrame

In [7]:
# Step 7: Create the final DataFrame with all data

print(f"\n{'='*60}")
print("CREATING FINAL DATAFRAME")
print(f"{'='*60}\n")

# Create lists for each column
metric_names = []
prev_week_values = []
this_week_values = []
wow_values = []

# Populate lists in the correct order
for metric in metrics:
    metric_names.append(metric)
    prev_week_values.append(prev_week_data[metric])
    this_week_values.append(this_week_data[metric])
    wow_values.append(wow_changes[metric])

# Create DataFrame
results_df = pd.DataFrame({
    'Metric': metric_names,
    prev_week_range: prev_week_values,
    this_week_range: this_week_values,
    'WoW': wow_values
})

# Display the DataFrame
print("Sales Metrics Analysis Table:")
print(results_df.to_string(index=False))


CREATING FINAL DATAFRAME

Sales Metrics Analysis Table:
                       Metric 11/23/2025 - 11/29/2025 11/30/2025 - 12/06/2025      WoW
                       Visits                    3294                    4010  +21.74%
                        CVR %                   3.94%                   3.29%  -16.50%
 Partial Typeform Submissions                                                         
Complete Typeform Submissions                                                         
       Booked Calls (Closers)                      83                     131  +57.83%
              Follow-up Calls                    28.0                    25.0  -10.71%
                    S2C Calls                    16.0                    30.0  +87.50%
    Total Closer Calls Booked                   127.0                   186.0  +46.46%
          Closer Availability                   115.0                   247.0 +114.78%
             Closer Fill Rate                 110.43%                  75

### Diagnose Blank Values

In [8]:
# Step 7.5: Diagnose why certain values are blank
# This will help identify the root cause of missing data

print(f"\n{'='*80}")
print("DIAGNOSING BLANK VALUES")
print(f"{'='*80}\n")

def diagnose_blank_value(csv_path, date_range, column_name, metric_name):
    """
    Diagnose why a value is blank by checking:
    1. Does the file exist?
    2. Does the date range exist in the file?
    3. Does the column exist?
    4. Is the cell value empty/blank?
    """
    diagnosis = {
        'metric': metric_name,
        'file': csv_path,
        'issue': None,
        'details': None
    }
    
    try:
        # Check if file exists
        if not os.path.exists(csv_path):
            diagnosis['issue'] = 'FILE_NOT_FOUND'
            diagnosis['details'] = f"File does not exist: {csv_path}"
            return diagnosis
        
        # Read the CSV
        df = pd.read_csv(csv_path)
        
        # Check if Date Range column exists
        if 'Date Range' not in df.columns:
            diagnosis['issue'] = 'MISSING_DATE_RANGE_COLUMN'
            diagnosis['details'] = f"'Date Range' column not found. Available columns: {list(df.columns)}"
            return diagnosis
        
        # Check if the specific column exists
        if column_name not in df.columns:
            diagnosis['issue'] = 'MISSING_DATA_COLUMN'
            diagnosis['details'] = f"Column '{column_name}' not found. Available columns: {list(df.columns)}"
            return diagnosis
        
        # Check if date range exists in the data
        row = df[df['Date Range'] == date_range]
        if row.empty:
            diagnosis['issue'] = 'DATE_RANGE_NOT_FOUND'
            diagnosis['details'] = f"Date range '{date_range}' not found in file. Available date ranges: {df['Date Range'].tolist()}"
            return diagnosis
        
        # Check if the value is blank/empty
        value = row.iloc[0][column_name]
        if pd.isna(value) or value == '' or value == ' ':
            diagnosis['issue'] = 'BLANK_CELL_VALUE'
            diagnosis['details'] = f"Cell exists but value is empty/blank/NaN for date range '{date_range}'"
            return diagnosis
        
        # If we get here, the value should exist
        diagnosis['issue'] = 'NO_ISSUE'
        diagnosis['details'] = f"Value found: {value}"
        return diagnosis
        
    except Exception as e:
        diagnosis['issue'] = 'ERROR'
        diagnosis['details'] = f"Error reading file: {str(e)}"
        return diagnosis


def diagnose_blank_excel_value(excel_path, sheet_name, date_range, row_name, metric_name):
    """
    Diagnose why an Excel value is blank
    """
    diagnosis = {
        'metric': metric_name,
        'file': excel_path,
        'issue': None,
        'details': None
    }
    
    try:
        # Check if file exists
        if not os.path.exists(excel_path):
            diagnosis['issue'] = 'FILE_NOT_FOUND'
            diagnosis['details'] = f"File does not exist: {excel_path}"
            return diagnosis
        
        # Read the Excel file
        df = pd.read_excel(excel_path, sheet_name=sheet_name)
        
        # Check if date_range column exists
        if date_range not in df.columns:
            diagnosis['issue'] = 'DATE_RANGE_COLUMN_NOT_FOUND'
            diagnosis['details'] = f"Date range '{date_range}' not found as column. Available columns: {list(df.columns)}"
            return diagnosis
        
        # Set first column as index
        df = df.set_index(df.columns[0])
        
        # Check if row exists
        if row_name not in df.index:
            diagnosis['issue'] = 'ROW_NOT_FOUND'
            diagnosis['details'] = f"Row '{row_name}' not found. Available rows: {df.index.tolist()}"
            return diagnosis
        
        # Check if value is blank
        value = df.loc[row_name, date_range]
        if pd.isna(value) or value == '' or value == ' ':
            diagnosis['issue'] = 'BLANK_CELL_VALUE'
            diagnosis['details'] = f"Cell exists but value is empty/blank/NaN"
            return diagnosis
        
        # If we get here, value should exist
        diagnosis['issue'] = 'NO_ISSUE'
        diagnosis['details'] = f"Value found: {value}"
        return diagnosis
        
    except Exception as e:
        diagnosis['issue'] = 'ERROR'
        diagnosis['details'] = f"Error reading file: {str(e)}"
        return diagnosis


# Create list to store all diagnoses
all_diagnoses = []

# Check each metric for both weeks
print("Checking PREVIOUS WEEK data sources:")
print("-" * 80)

# 1. Visits
if prev_week_data['Visits'] == '':
    diag = diagnose_blank_value('../outputs/pro_site/weekly_pro_site.csv', prev_week_range, 'Visits', 'Visits (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 2. CVR %
if prev_week_data['CVR %'] == '':
    diag = diagnose_blank_value('../outputs/pro_site/weekly_pro_site.csv', prev_week_range, 'Conversion Rate', 'CVR % (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 3. Partial Typeform Submissions
if prev_week_data['Partial Typeform Submissions'] == '':
    diag = diagnose_blank_value('../outputs/typeform_submissions/weekly_typeform_submissions.csv', prev_week_range, 'Partials', 'Partial Typeform Submissions (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 4. Complete Typeform Submissions
if prev_week_data['Complete Typeform Submissions'] == '':
    diag = diagnose_blank_value('../outputs/typeform_submissions/weekly_typeform_submissions.csv', prev_week_range, 'Completed', 'Complete Typeform Submissions (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 5. Booked Calls (Closers)
if prev_week_data['Booked Calls (Closers)'] == '':
    diag = diagnose_blank_value('../outputs/discovery_intro/weekly_discovery_call.csv', prev_week_range, 'Booked Calls (Completed)', 'Booked Calls (Closers) (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 6. Follow-up Calls
if prev_week_data['Follow-up Calls'] == '':
    diag = diagnose_blank_excel_value('../data/stat.xlsx', 'stat', prev_week_range, 'Follow ups', 'Follow-up Calls (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 7. S2C Calls
if prev_week_data['S2C Calls'] == '':
    diag = diagnose_blank_excel_value('../data/stat.xlsx', 'stat', prev_week_range, 'S2C', 'S2C Calls (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 8. Closer Availability
if prev_week_data['Closer Availability'] == '':
    diag = diagnose_blank_excel_value('../data/stat.xlsx', 'stat', prev_week_range, 'Capacity', 'Closer Availability (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 9. Booked Calls (Setters)
if prev_week_data['Booked Calls (Setters)'] == '':
    diag = diagnose_blank_value('../outputs/discovery_intro/weekly_intro_call.csv', prev_week_range, 'Booked Calls (Completed)', 'Booked Calls (Setters) (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 10. Scheduled Calls
if prev_week_data['Scheduled Calls'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', prev_week_range, 'Sched. calls', 'Scheduled Calls (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 11. Live Calls
if prev_week_data['Live Calls'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', prev_week_range, 'Live calls', 'Live Calls (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 12. Show Rate
if prev_week_data['Show Rate'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', prev_week_range, 'Show %', 'Show Rate (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 13. Avg. Days Booked Out
if prev_week_data['Avg. Days Booked Out'] == '':
    diag = diagnose_blank_value('../outputs/discovery_intro/weekly_discovery_intro_blended.csv', prev_week_range, 'Avg Lead Time (Days)(Completed)', 'Avg. Days Booked Out (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 14. Offers
if prev_week_data['Offers'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', prev_week_range, 'Offers', 'Offers (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 15. Offer Rate
if prev_week_data['Offer Rate'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', prev_week_range, 'Offer %', 'Offer Rate (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 16. Closes
if prev_week_data['Closes'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', prev_week_range, 'Close 1', 'Closes (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 17. Offer to Close
if prev_week_data['Offer to Close'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', prev_week_range, 'Offer to Close %', 'Offer to Close (Prev)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# Now check THIS WEEK
print("\n" + "="*80)
print("Checking THIS WEEK data sources:")
print("-" * 80)

# Repeat same checks for this week
# 1. Visits
if this_week_data['Visits'] == '':
    diag = diagnose_blank_value('../outputs/pro_site/weekly_pro_site.csv', this_week_range, 'Visits', 'Visits (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 2. CVR %
if this_week_data['CVR %'] == '':
    diag = diagnose_blank_value('../outputs/pro_site/weekly_pro_site.csv', this_week_range, 'Conversion Rate', 'CVR % (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 3. Partial Typeform Submissions
if this_week_data['Partial Typeform Submissions'] == '':
    diag = diagnose_blank_value('../outputs/typeform_submissions/weekly_typeform_submissions.csv', this_week_range, 'Partials', 'Partial Typeform Submissions (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 4. Complete Typeform Submissions
if this_week_data['Complete Typeform Submissions'] == '':
    diag = diagnose_blank_value('../outputs/typeform_submissions/weekly_typeform_submissions.csv', this_week_range, 'Completed', 'Complete Typeform Submissions (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 5. Booked Calls (Closers)
if this_week_data['Booked Calls (Closers)'] == '':
    diag = diagnose_blank_value('../outputs/discovery_intro/weekly_discovery_call.csv', this_week_range, 'Booked Calls (Completed)', 'Booked Calls (Closers) (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 6. Follow-up Calls
if this_week_data['Follow-up Calls'] == '':
    diag = diagnose_blank_excel_value('../data/stat.xlsx', 'stat', this_week_range, 'Follow ups', 'Follow-up Calls (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 7. S2C Calls
if this_week_data['S2C Calls'] == '':
    diag = diagnose_blank_excel_value('../data/stat.xlsx', 'stat', this_week_range, 'S2C', 'S2C Calls (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 8. Closer Availability
if this_week_data['Closer Availability'] == '':
    diag = diagnose_blank_excel_value('../data/stat.xlsx', 'stat', this_week_range, 'Capacity', 'Closer Availability (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 9. Booked Calls (Setters)
if this_week_data['Booked Calls (Setters)'] == '':
    diag = diagnose_blank_value('../outputs/discovery_intro/weekly_intro_call.csv', this_week_range, 'Booked Calls (Completed)', 'Booked Calls (Setters) (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 10. Scheduled Calls
if this_week_data['Scheduled Calls'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', this_week_range, 'Sched. calls', 'Scheduled Calls (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 11. Live Calls
if this_week_data['Live Calls'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', this_week_range, 'Live calls', 'Live Calls (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 12. Show Rate
if this_week_data['Show Rate'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', this_week_range, 'Show %', 'Show Rate (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 13. Avg. Days Booked Out
if this_week_data['Avg. Days Booked Out'] == '':
    diag = diagnose_blank_value('../outputs/discovery_intro/weekly_discovery_intro_blended.csv', this_week_range, 'Avg Lead Time (Days)(Completed)', 'Avg. Days Booked Out (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 14. Offers
if this_week_data['Offers'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', this_week_range, 'Offers', 'Offers (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 15. Offer Rate
if this_week_data['Offer Rate'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', this_week_range, 'Offer %', 'Offer Rate (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 16. Closes
if this_week_data['Closes'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', this_week_range, 'Close 1', 'Closes (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# 17. Offer to Close
if this_week_data['Offer to Close'] == '':
    diag = diagnose_blank_value('../outputs/sales_tracker/weekly_sales_tracker.csv', this_week_range, 'Offer to Close %', 'Offer to Close (This)')
    all_diagnoses.append(diag)
    print(f"❌ {diag['metric']}: {diag['issue']}")
    print(f"   → {diag['details']}\n")

# Summary
print("\n" + "="*80)
print("DIAGNOSIS SUMMARY")
print("="*80)

if len(all_diagnoses) == 0:
    print("✅ No blank values found - all data loaded successfully!")
else:
    print(f"Found {len(all_diagnoses)} blank value(s)")
    print("\nIssue breakdown:")
    issue_counts = {}
    for d in all_diagnoses:
        issue = d['issue']
        if issue in issue_counts:
            issue_counts[issue] += 1
        else:
            issue_counts[issue] = 1
    
    for issue, count in issue_counts.items():
        print(f"  - {issue}: {count} occurrence(s)")

print("\n" + "="*80)


DIAGNOSING BLANK VALUES

Checking PREVIOUS WEEK data sources:
--------------------------------------------------------------------------------
❌ Partial Typeform Submissions (Prev): DATE_RANGE_NOT_FOUND
   → Date range '11/23/2025 - 11/29/2025' not found in file. Available date ranges: ['10/03/2025 - 10/04/2025', '10/05/2025 - 10/11/2025', '10/12/2025 - 10/18/2025', '10/19/2025 - 10/25/2025', '10/26/2025 - 11/01/2025', '11/02/2025 - 11/07/2025']

❌ Complete Typeform Submissions (Prev): DATE_RANGE_NOT_FOUND
   → Date range '11/23/2025 - 11/29/2025' not found in file. Available date ranges: ['10/03/2025 - 10/04/2025', '10/05/2025 - 10/11/2025', '10/12/2025 - 10/18/2025', '10/19/2025 - 10/25/2025', '10/26/2025 - 11/01/2025', '11/02/2025 - 11/07/2025']


Checking THIS WEEK data sources:
--------------------------------------------------------------------------------
❌ Partial Typeform Submissions (This): DATE_RANGE_NOT_FOUND
   → Date range '11/30/2025 - 12/06/2025' not found in file. Ava

### Create Output Folder and Save as CSV

In [9]:
# Step 8: Create output folder and save results as CSV

print(f"\n{'='*60}")
print("SAVING RESULTS TO CSV")
print(f"{'='*60}\n")

# Define output folder path (relative to notebooks folder)
output_folder = '../outputs/sales_metrics_analysis'

# Create folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Created folder: {output_folder}")
else:
    print(f"Folder already exists: {output_folder}")

# Define output file path
output_csv_path = os.path.join(output_folder, 'weekly_sales_metrics.csv')

# Save DataFrame to CSV
results_df.to_csv(output_csv_path, index=False)

print(f"CSV saved successfully to: {output_csv_path}")


SAVING RESULTS TO CSV

Folder already exists: ../outputs/sales_metrics_analysis
CSV saved successfully to: ../outputs/sales_metrics_analysis/weekly_sales_metrics.csv


### Apply Color Coding to Excel

In [10]:
# Step 9: Apply color coding to the WoW column in Excel format
# Note: We'll save as Excel (.xlsx) with color formatting

print(f"\n{'='*60}")
print("APPLYING COLOR CODING (EXCEL FORMAT)")
print(f"{'='*60}\n")

# Define output Excel file path
output_excel_path = os.path.join(output_folder, 'weekly_sales_metrics.xlsx')

# Save DataFrame to Excel first
results_df.to_excel(output_excel_path, index=False, sheet_name='Sales Metrics')

# Load the workbook to apply formatting
wb = load_workbook(output_excel_path)
ws = wb['Sales Metrics']

# Find the WoW column (it's the 4th column, index D)
wow_col_idx = 4  # Column D (1=A, 2=B, 3=C, 4=D)

# Apply color coding to WoW column (starting from row 2, as row 1 is header)
for row in range(2, len(results_df) + 2):  # +2 because Excel rows start at 1 and we skip header
    cell = ws.cell(row=row, column=wow_col_idx)
    cell_value = cell.value
    
    # Check if cell has a value and is not blank
    if cell_value and cell_value != '':
        # Convert to string to check for + or - sign
        cell_str = str(cell_value)
        
        if cell_str.startswith('+'):
            # Positive change - Green color
            cell.font = Font(color="00B050", bold=True)  # Green
        elif cell_str.startswith('-'):
            # Negative change - Red color
            cell.font = Font(color="FF0000", bold=True)  # Red

# Save the formatted workbook
wb.save(output_excel_path)

print(f"Excel file with color coding saved to: {output_excel_path}")
print("\nNote: CSV file does not support color formatting.")
print("Use the Excel file for viewing with colors, or the CSV for data processing.")


APPLYING COLOR CODING (EXCEL FORMAT)

Excel file with color coding saved to: ../outputs/sales_metrics_analysis/weekly_sales_metrics.xlsx

Note: CSV file does not support color formatting.
Use the Excel file for viewing with colors, or the CSV for data processing.


### Summary and Verification

In [11]:
# Step 10: Display summary and final verification

print(f"\n{'='*80}")
print("ANALYSIS COMPLETE!")
print(f"{'='*80}\n")

print(f"Analysis Period:")
print(f"  Previous Week: {prev_week_range}")
print(f"  This Week:     {this_week_range}")
print(f"\nOutput Files Created:")
print(f"  1. CSV (no colors):  {output_csv_path}")
print(f"  2. Excel (colored):  {output_excel_path}")
print(f"\nTotal Metrics Analyzed: {len(results_df)}")
print(f"\nMetrics with WoW Increase: {len([w for w in wow_values if w.startswith('+') if w])}")
print(f"Metrics with WoW Decrease: {len([w for w in wow_values if w.startswith('-') if w])}")
print(f"Metrics with Missing Data: {len([w for w in wow_values if w == ''])}")

print("\n" + "="*80)
print("You can now use these files for your Word document report!")
print("="*80)


ANALYSIS COMPLETE!

Analysis Period:
  Previous Week: 11/23/2025 - 11/29/2025
  This Week:     11/30/2025 - 12/06/2025

Output Files Created:
  1. CSV (no colors):  ../outputs/sales_metrics_analysis/weekly_sales_metrics.csv
  2. Excel (colored):  ../outputs/sales_metrics_analysis/weekly_sales_metrics.xlsx

Total Metrics Analyzed: 19

Metrics with WoW Increase: 10
Metrics with WoW Decrease: 7
Metrics with Missing Data: 2

You can now use these files for your Word document report!


#

## MONTHLY TABLE

### Load Monthly Reference Dates

In [12]:
# Step 1: Load the Custom Start Date and Custom End Date for Monthly analysis

custom_start_date = None
custom_end_date = None

# Re-read the analysis_ref_date.csv file to get Custom dates
with open(ref_date_path, 'r') as f:
    lines = f.readlines()

for line in lines:
    # Look for Custom Start Date
    if 'Custom Start Date' in line and custom_start_date is None:
        parts = line.strip().split(',')
        if len(parts) >= 2:
            custom_start_date = parts[1].strip()
    
    # Look for Custom End Date
    if 'Custom End Date' in line and custom_end_date is None:
        parts = line.strip().split(',')
        if len(parts) >= 2:
            custom_end_date = parts[1].strip()
    
    # Break once we have both dates
    if custom_start_date and custom_end_date:
        break

print("Monthly Reference Dates Loaded:")
print(f"Custom Start Date: {custom_start_date}")
print(f"Custom End Date: {custom_end_date}")

# Create "This Month" date range string EXACTLY as it appears in CSV files
# Format: MM/DD/YYYY - MM/DD/YYYY
this_month_range = f"{custom_start_date} - {custom_end_date}"
print(f"\nThis Month Range: {this_month_range}")

# Calculate "Previous Month" range
# Parse the dates to datetime objects
this_month_start = pd.to_datetime(custom_start_date, format='%m/%d/%Y')
this_month_end = pd.to_datetime(custom_end_date, format='%m/%d/%Y')

# Previous month: go back one month
prev_month_start = this_month_start - pd.DateOffset(months=1)
prev_month_end = this_month_end - pd.DateOffset(months=1)

# Format back to MM/DD/YYYY - MM/DD/YYYY (with leading zeros maintained)
prev_month_range = f"{prev_month_start.strftime('%m/%d/%Y')} - {prev_month_end.strftime('%m/%d/%Y')}"

print(f"Previous Month Range: {prev_month_range}")

# Calculate days in period
days_in_period = (this_month_end - this_month_start).days + 1
print(f"\nDays in analysis period: {days_in_period}")

Monthly Reference Dates Loaded:
Custom Start Date: 11/01/2025
Custom End Date: 11/28/2025

This Month Range: 11/01/2025 - 11/28/2025
Previous Month Range: 10/01/2025 - 10/28/2025

Days in analysis period: 28


### Define Monthly Helper Function

In [13]:
# Step 2: Define helper function for monthly calculations

def calculate_mom_change(current_value, previous_value):
    """
    Calculate Month-over-Month (MoM) or Month-to-Date (MTD) percentage change
    
    Formula: (Current - Previous) / Previous * 100
    """
    try:
        # Check if either value is blank/empty
        if current_value == '' or previous_value == '' or current_value is None or previous_value is None:
            return ''
        
        # Convert to float (handle percentage strings)
        if isinstance(current_value, str):
            current_value = current_value.replace('%', '').strip()
            if current_value == '':
                return ''
            current_value = float(current_value)
        
        if isinstance(previous_value, str):
            previous_value = previous_value.replace('%', '').strip()
            if previous_value == '':
                return ''
            previous_value = float(previous_value)
        
        # Avoid division by zero
        if previous_value == 0:
            return ''
        
        # Calculate percentage change
        mom_change = ((current_value - previous_value) / previous_value) * 100
        
        # Format with + or - sign
        if mom_change >= 0:
            return f"+{mom_change:.2f}%"
        else:
            return f"{mom_change:.2f}%"
    
    except Exception as e:
        print(f"Error calculating MoM change: {str(e)}")
        return ''

print("Monthly helper function defined successfully!")

Monthly helper function defined successfully!


### Extract Monthly Data

In [14]:
# Step 3: Extract data for both This Month and Previous Month

print("=" * 70)
print("EXTRACTING MONTHLY DATA FROM ALL SOURCES")
print("=" * 70)

# Define all CSV file paths
csv_sources = {
    'newsletter': '../outputs/newsletter/custom_newsletter.csv',
    'pro_site': '../outputs/pro_site/custom_pro_site.csv',
    'typeform': '../outputs/typeform_submissions/custom_typeform_submissions.csv',
    'discovery_call': '../outputs/discovery_intro/custom_discovery_call.csv',
    'intro_call': '../outputs/discovery_intro/custom_intro_call.csv',
    'sales_tracker': '../outputs/sales_tracker/custom_sales_tracker.csv',
    'discovery_intro_blended': '../outputs/discovery_intro/custom_discovery_intro_blended.csv'
}

# Initialize dictionary
monthly_data = {}

print(f"\nThis Month: {this_month_range}")
print(f"Previous Month: {prev_month_range}")
print("\n" + "-" * 70)

# 1. Visits
print("\n1. Extracting Visits...")
this_pro_lp_visits = get_value_from_csv(csv_sources['pro_site'], this_month_range, 'Visits')
prev_pro_lp_visits = get_value_from_csv(csv_sources['pro_site'], prev_month_range, 'Visits')
monthly_data['Visits'] = {'this_month': this_pro_lp_visits, 'prev_month': prev_pro_lp_visits}
print(f"   This Month: {this_pro_lp_visits}")
print(f"   Previous Month: {prev_pro_lp_visits}")

# 2. CVR %
print("\n2. Extracting CVR %...")
this_pro_lp_cvr = get_value_from_csv(csv_sources['pro_site'], this_month_range, 'Conversion Rate')
prev_pro_lp_cvr = get_value_from_csv(csv_sources['pro_site'], prev_month_range, 'Conversion Rate')
monthly_data['CVR %'] = {'this_month': this_pro_lp_cvr, 'prev_month': prev_pro_lp_cvr}
print(f"   This Month: {this_pro_lp_cvr}")
print(f"   Previous Month: {prev_pro_lp_cvr}")

# 3. Partial Typeform Submissions
print("\n3. Extracting Partial Typeform Submissions...")
this_partial = get_value_from_csv(csv_sources['typeform'], this_month_range, 'Partials')
prev_partial = get_value_from_csv(csv_sources['typeform'], prev_month_range, 'Partials')
monthly_data['Partial Typeform Submissions'] = {'this_month': this_partial, 'prev_month': prev_partial}
print(f"   This Month: {this_partial}")
print(f"   Previous Month: {prev_partial}")

# 4. Complete Typeform Submissions
print("\n4. Extracting Complete Typeform Submissions...")
this_complete = get_value_from_csv(csv_sources['typeform'], this_month_range, 'Completed')
prev_complete = get_value_from_csv(csv_sources['typeform'], prev_month_range, 'Completed')
monthly_data['Complete Typeform Submissions'] = {'this_month': this_complete, 'prev_month': prev_complete}
print(f"   This Month: {this_complete}")
print(f"   Previous Month: {prev_complete}")

# 5. Booked Calls (Closers)
print("\n5. Extracting Booked Calls (Closers)...")
this_closers = get_value_from_csv(csv_sources['discovery_call'], this_month_range, 'Booked Calls (Completed)')
prev_closers = get_value_from_csv(csv_sources['discovery_call'], prev_month_range, 'Booked Calls (Completed)')
monthly_data['Booked Calls (Closers)'] = {'this_month': this_closers, 'prev_month': prev_closers}
print(f"   This Month: {this_closers}")
print(f"   Previous Month: {prev_closers}")

# 6. Booked Calls (Setters)
print("\n6. Extracting Booked Calls (Setters)...")
this_setters = get_value_from_csv(csv_sources['intro_call'], this_month_range, 'Booked Calls (Completed)')
prev_setters = get_value_from_csv(csv_sources['intro_call'], prev_month_range, 'Booked Calls (Completed)')
monthly_data['Booked Calls (Setters)'] = {'this_month': this_setters, 'prev_month': prev_setters}
print(f"   This Month: {this_setters}")
print(f"   Previous Month: {prev_setters}")

# 7. Scheduled Calls
print("\n7. Extracting Scheduled Calls...")
this_scheduled = get_value_from_csv(csv_sources['sales_tracker'], this_month_range, 'Sched. calls')
prev_scheduled = get_value_from_csv(csv_sources['sales_tracker'], prev_month_range, 'Sched. calls')
monthly_data['Scheduled Calls'] = {'this_month': this_scheduled, 'prev_month': prev_scheduled}
print(f"   This Month: {this_scheduled}")
print(f"   Previous Month: {prev_scheduled}")

# 8. Live Calls
print("\n8. Extracting Live Calls...")
this_live = get_value_from_csv(csv_sources['sales_tracker'], this_month_range, 'Live calls')
prev_live = get_value_from_csv(csv_sources['sales_tracker'], prev_month_range, 'Live calls')
monthly_data['Live Calls'] = {'this_month': this_live, 'prev_month': prev_live}
print(f"   This Month: {this_live}")
print(f"   Previous Month: {prev_live}")

# 9. Show Rate
print("\n9. Extracting Show Rate...")
this_show_rate = get_value_from_csv(csv_sources['sales_tracker'], this_month_range, 'Show %')
prev_show_rate = get_value_from_csv(csv_sources['sales_tracker'], prev_month_range, 'Show %')
monthly_data['Show Rate'] = {'this_month': this_show_rate, 'prev_month': prev_show_rate}
print(f"   This Month: {this_show_rate}")
print(f"   Previous Month: {prev_show_rate}")

# 10. Average Lead Time
print("\n10. Extracting Average Lead Time...")
this_avg_lead = get_value_from_csv(csv_sources['discovery_intro_blended'], this_month_range, 'Avg Lead Time (Days)(Completed)')
prev_avg_lead = get_value_from_csv(csv_sources['discovery_intro_blended'], prev_month_range, 'Avg Lead Time (Days)(Completed)')
monthly_data['Average Lead Time'] = {'this_month': this_avg_lead, 'prev_month': prev_avg_lead}
print(f"   This Month: {this_avg_lead}")
print(f"   Previous Month: {prev_avg_lead}")

# 11. Offers
print("\n11. Extracting Offers...")
this_offers = get_value_from_csv(csv_sources['sales_tracker'], this_month_range, 'Offers')
prev_offers = get_value_from_csv(csv_sources['sales_tracker'], prev_month_range, 'Offers')
monthly_data['Offers'] = {'this_month': this_offers, 'prev_month': prev_offers}
print(f"   This Month: {this_offers}")
print(f"   Previous Month: {prev_offers}")

# 12. Offer Rate
print("\n12. Extracting Offer Rate...")
this_offer_rate = get_value_from_csv(csv_sources['sales_tracker'], this_month_range, 'Offer %')
prev_offer_rate = get_value_from_csv(csv_sources['sales_tracker'], prev_month_range, 'Offer %')
monthly_data['Offer Rate'] = {'this_month': this_offer_rate, 'prev_month': prev_offer_rate}
print(f"   This Month: {this_offer_rate}")
print(f"   Previous Month: {prev_offer_rate}")

# 13. Closes
print("\n13. Extracting Closes...")
this_closes = get_value_from_csv(csv_sources['sales_tracker'], this_month_range, 'Close 1')
prev_closes = get_value_from_csv(csv_sources['sales_tracker'], prev_month_range, 'Close 1')
monthly_data['Closes'] = {'this_month': this_closes, 'prev_month': prev_closes}
print(f"   This Month: {this_closes}")
print(f"   Previous Month: {prev_closes}")

# 14. Offer to Close
print("\n14. Extracting Offer to Close...")
this_offer_to_close = get_value_from_csv(csv_sources['sales_tracker'], this_month_range, 'Offer to Close %')
prev_offer_to_close = get_value_from_csv(csv_sources['sales_tracker'], prev_month_range, 'Offer to Close %')
monthly_data['Offer to Close'] = {'this_month': this_offer_to_close, 'prev_month': prev_offer_to_close}
print(f"   This Month: {this_offer_to_close}")
print(f"   Previous Month: {prev_offer_to_close}")

print("\n" + "=" * 70)
print("DATA EXTRACTION COMPLETE")
print("=" * 70)

EXTRACTING MONTHLY DATA FROM ALL SOURCES

This Month: 11/01/2025 - 11/28/2025
Previous Month: 10/01/2025 - 10/28/2025

----------------------------------------------------------------------

1. Extracting Visits...
   This Month: 15163
   Previous Month: 10528

2. Extracting CVR %...
   This Month: 3.33%
   Previous Month: 4.0%

3. Extracting Partial Typeform Submissions...
   This Month: 
   Previous Month: 

4. Extracting Complete Typeform Submissions...
   This Month: 
   Previous Month: 

5. Extracting Booked Calls (Closers)...
   This Month: 403
   Previous Month: 315

6. Extracting Booked Calls (Setters)...
   This Month: 144
   Previous Month: 89

7. Extracting Scheduled Calls...
   This Month: 493.0
   Previous Month: 408.0

8. Extracting Live Calls...
   This Month: 338.0
   Previous Month: 306.0

9. Extracting Show Rate...
   This Month: 56.09%
   Previous Month: 70.65%

10. Extracting Average Lead Time...
   This Month: 3.8
   Previous Month: 2.53

11. Extracting Offers...
 

### Diagnose Blank Values

In [15]:
# Step 3.5: Diagnose why values might be blank

print(f"\n{'='*80}")
print("DIAGNOSING BLANK VALUES FOR MONTHLY DATA")
print(f"{'='*80}\n")

def diagnose_monthly_blank(csv_path, date_range, column_name, metric_name):
    """
    Diagnose why a monthly value is blank
    """
    diagnosis = {
        'metric': metric_name,
        'file': csv_path,
        'issue': None,
        'details': None
    }
    
    try:
        if not os.path.exists(csv_path):
            diagnosis['issue'] = 'FILE_NOT_FOUND'
            diagnosis['details'] = f"File does not exist: {csv_path}"
            return diagnosis
        
        df = pd.read_csv(csv_path)
        
        if 'Date Range' not in df.columns:
            diagnosis['issue'] = 'MISSING_DATE_RANGE_COLUMN'
            diagnosis['details'] = f"'Date Range' column not found. Available columns: {list(df.columns)}"
            return diagnosis
        
        if column_name not in df.columns:
            diagnosis['issue'] = 'MISSING_DATA_COLUMN'
            diagnosis['details'] = f"Column '{column_name}' not found. Available columns: {list(df.columns)}"
            return diagnosis
        
        row = df[df['Date Range'] == date_range]
        if row.empty:
            diagnosis['issue'] = 'DATE_RANGE_NOT_FOUND'
            diagnosis['details'] = f"Date range '{date_range}' not found. Available ranges: {df['Date Range'].tolist()}"
            return diagnosis
        
        value = row.iloc[0][column_name]
        if pd.isna(value) or value == '':
            diagnosis['issue'] = 'BLANK_CELL_VALUE'
            diagnosis['details'] = f"Cell exists but value is empty/blank for '{date_range}'"
            return diagnosis
        
        diagnosis['issue'] = 'NO_ISSUE'
        diagnosis['details'] = f"Value found: {value}"
        return diagnosis
        
    except Exception as e:
        diagnosis['issue'] = 'ERROR'
        diagnosis['details'] = f"Error: {str(e)}"
        return diagnosis

# Check metrics with blank values
blank_metrics = []
issue_summary = {}

print("Checking PREVIOUS MONTH data sources:")
print("-" * 80)

for metric_name, values in monthly_data.items():
    if values['prev_month'] == '' or values['prev_month'] is None:
        # Find the source for this metric
        if metric_name == 'Visits':
            diagnosis = diagnose_monthly_blank(csv_sources['newsletter'], prev_month_range, 'Visits', f"{metric_name} (Prev)")
        elif metric_name == 'CVR %':
            diagnosis = diagnose_monthly_blank(csv_sources['pro_site'], prev_month_range, 'Conversion Rate', f"{metric_name} (Prev)")
        elif metric_name == 'Partial Typeform Submissions':
            diagnosis = diagnose_monthly_blank(csv_sources['typeform'], prev_month_range, 'Partials', f"{metric_name} (Prev)")
        elif metric_name == 'Complete Typeform Submissions':
            diagnosis = diagnose_monthly_blank(csv_sources['typeform'], prev_month_range, 'Completed', f"{metric_name} (Prev)")
        elif metric_name == 'Booked Calls (Closers)':
            diagnosis = diagnose_monthly_blank(csv_sources['discovery_call'], prev_month_range, 'Booked Calls (Completed)', f"{metric_name} (Prev)")
        elif metric_name == 'Booked Calls (Setters)':
            diagnosis = diagnose_monthly_blank(csv_sources['intro_call'], prev_month_range, 'Booked Calls (Completed)', f"{metric_name} (Prev)")
        elif metric_name in ['Scheduled Calls', 'Live Calls', 'Show Rate', 'Offers', 'Offer Rate', 'Closes', 'Offer to Close']:
            column_map = {
                'Scheduled Calls': 'Sched. calls',
                'Live Calls': 'Live calls',
                'Show Rate': 'Show %',
                'Offers': 'Offers',
                'Offer Rate': 'Offer %',
                'Closes': 'Close 1',
                'Offer to Close': 'Offer to Close %'
            }
            diagnosis = diagnose_monthly_blank(csv_sources['sales_tracker'], prev_month_range, column_map[metric_name], f"{metric_name} (Prev)")
        elif metric_name == 'Average Lead Time':
            diagnosis = diagnose_monthly_blank(csv_sources['discovery_intro_blended'], prev_month_range, 'Avg Lead Time (Days)(Completed)', f"{metric_name} (Prev)")
        else:
            continue
        
        blank_metrics.append(diagnosis)
        issue_summary[diagnosis['issue']] = issue_summary.get(diagnosis['issue'], 0) + 1
        
        print(f"❌ {diagnosis['metric']}: {diagnosis['issue']}")
        print(f"   → {diagnosis['details']}\n")

print("\n" + "=" * 80)
print("Checking THIS MONTH data sources:")
print("-" * 80)

for metric_name, values in monthly_data.items():
    if values['this_month'] == '' or values['this_month'] is None:
        # Find the source for this metric
        if metric_name == 'Visits':
            diagnosis = diagnose_monthly_blank(csv_sources['newsletter'], this_month_range, 'Visits', f"{metric_name} (This)")
        elif metric_name == 'CVR %':
            diagnosis = diagnose_monthly_blank(csv_sources['pro_site'], this_month_range, 'Conversion Rate', f"{metric_name} (This)")
        elif metric_name == 'Partial Typeform Submissions':
            diagnosis = diagnose_monthly_blank(csv_sources['typeform'], this_month_range, 'Partials', f"{metric_name} (This)")
        elif metric_name == 'Complete Typeform Submissions':
            diagnosis = diagnose_monthly_blank(csv_sources['typeform'], this_month_range, 'Completed', f"{metric_name} (This)")
        elif metric_name == 'Booked Calls (Closers)':
            diagnosis = diagnose_monthly_blank(csv_sources['discovery_call'], this_month_range, 'Booked Calls (Completed)', f"{metric_name} (This)")
        elif metric_name == 'Booked Calls (Setters)':
            diagnosis = diagnose_monthly_blank(csv_sources['intro_call'], this_month_range, 'Booked Calls (Completed)', f"{metric_name} (This)")
        elif metric_name in ['Scheduled Calls', 'Live Calls', 'Show Rate', 'Offers', 'Offer Rate', 'Closes', 'Offer to Close']:
            column_map = {
                'Scheduled Calls': 'Sched. calls',
                'Live Calls': 'Live calls',
                'Show Rate': 'Show %',
                'Offers': 'Offers',
                'Offer Rate': 'Offer %',
                'Closes': 'Close 1',
                'Offer to Close': 'Offer to Close %'
            }
            diagnosis = diagnose_monthly_blank(csv_sources['sales_tracker'], this_month_range, column_map[metric_name], f"{metric_name} (This)")
        elif metric_name == 'Average Lead Time':
            diagnosis = diagnose_monthly_blank(csv_sources['discovery_intro_blended'], this_month_range, 'Avg Lead Time (Days)(Completed)', f"{metric_name} (This)")
        else:
            continue
        
        blank_metrics.append(diagnosis)
        issue_summary[diagnosis['issue']] = issue_summary.get(diagnosis['issue'], 0) + 1
        
        print(f"❌ {diagnosis['metric']}: {diagnosis['issue']}")
        print(f"   → {diagnosis['details']}\n")

print("\n" + "=" * 80)
print("DIAGNOSIS SUMMARY")
print("=" * 80)
print(f"Found {len(blank_metrics)} blank value(s)\n")

if issue_summary:
    print("Issue breakdown:")
    for issue_type, count in issue_summary.items():
        print(f"  - {issue_type}: {count} occurrence(s)")

print("\n" + "=" * 80)


DIAGNOSING BLANK VALUES FOR MONTHLY DATA

Checking PREVIOUS MONTH data sources:
--------------------------------------------------------------------------------
❌ Partial Typeform Submissions (Prev): DATE_RANGE_NOT_FOUND
   → Date range '10/01/2025 - 10/28/2025' not found. Available ranges: ['11/01/2025 - 11/07/2025', '10/01/2025 - 10/07/2025']

❌ Complete Typeform Submissions (Prev): DATE_RANGE_NOT_FOUND
   → Date range '10/01/2025 - 10/28/2025' not found. Available ranges: ['11/01/2025 - 11/07/2025', '10/01/2025 - 10/07/2025']


Checking THIS MONTH data sources:
--------------------------------------------------------------------------------
❌ Partial Typeform Submissions (This): DATE_RANGE_NOT_FOUND
   → Date range '11/01/2025 - 11/28/2025' not found. Available ranges: ['11/01/2025 - 11/07/2025', '10/01/2025 - 10/07/2025']

❌ Complete Typeform Submissions (This): DATE_RANGE_NOT_FOUND
   → Date range '11/01/2025 - 11/28/2025' not found. Available ranges: ['11/01/2025 - 11/07/2025', 

### Calculate MoM/MTD Changes

In [16]:
# Step 4: Calculate MoM/MTD Change for all metrics

print("\n" + "=" * 70)
print("CALCULATING MOM/MTD CHANGES")
print("=" * 70)

for metric_name, values in monthly_data.items():
    mom_change = calculate_mom_change(values['this_month'], values['prev_month'])
    monthly_data[metric_name]['mom_change'] = mom_change
    print(f"{metric_name}: {mom_change}")

print("\n" + "=" * 70)
print("CALCULATIONS COMPLETE")
print("=" * 70)


CALCULATING MOM/MTD CHANGES
Visits: +44.03%
CVR %: -16.75%
Partial Typeform Submissions: 
Complete Typeform Submissions: 
Booked Calls (Closers): +27.94%
Booked Calls (Setters): +61.80%
Scheduled Calls: +20.83%
Live Calls: +10.46%
Show Rate: -20.61%
Average Lead Time: +50.20%
Offers: +3.50%
Offer Rate: -20.33%
Closes: +3.08%
Offer to Close: +17.68%

CALCULATIONS COMPLETE


### Create Monthly Table

In [17]:
# Step 5: Create the monthly comparison table

print("\n" + "=" * 70)
print("CREATING MONTHLY COMPARISON TABLE")
print("=" * 70)

table_data = []

for metric_name in [
    'Visits',
    'CVR %',
    'Partial Typeform Submissions',
    'Complete Typeform Submissions',
    'Booked Calls (Closers)',
    'Booked Calls (Setters)',
    'Scheduled Calls',
    'Live Calls',
    'Show Rate',
    'Average Lead Time',
    'Offers',
    'Offer Rate',
    'Closes',
    'Offer to Close'
]:
    table_data.append({
        'Metric': metric_name,
        prev_month_range: monthly_data[metric_name]['prev_month'],
        this_month_range: monthly_data[metric_name]['this_month'],
        'MoM/MTD': monthly_data[metric_name]['mom_change']
    })

monthly_table = pd.DataFrame(table_data)

print("\nMONTHLY COMPARISON TABLE:")
print("=" * 70)
display(monthly_table)


CREATING MONTHLY COMPARISON TABLE

MONTHLY COMPARISON TABLE:


,Metric,10/01/2025 - 10/28/2025,11/01/2025 - 11/28/2025,MoM/MTD
0,Visits,10528,15163,+44.03%
1,CVR %,4.0%,3.33%,-16.75%
2,Partial Typeform Submissions,,,
3,Complete Typeform Submissions,,,
4,Booked Calls (Closers),315,403,+27.94%
5,Booked Calls (Setters),89,144,+61.80%
6,Scheduled Calls,408.0,493.0,+20.83%
7,Live Calls,306.0,338.0,+10.46%
8,Show Rate,70.65%,56.09%,-20.61%
9,Average Lead Time,2.53,3.8,+50.20%


### Save Monthly Table to CSV

In [18]:
# Step 6: Save the monthly comparison table to CSV

output_dir = '../outputs/sales_metrics_analysis'
output_file = os.path.join(output_dir, 'monthly_sales_metrics.csv')

os.makedirs(output_dir, exist_ok=True)

monthly_table.to_csv(output_file, index=False)

print(f"\n✓ Monthly comparison table saved to: {output_file}")


✓ Monthly comparison table saved to: ../outputs/sales_metrics_analysis/monthly_sales_metrics.csv


### Save Monthly Table to Excel with Color Coding

In [19]:
# Step 7: Save to Excel with color coding

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

excel_output_file = os.path.join(output_dir, 'monthly_sales_metrics.xlsx')

wb = Workbook()
ws = wb.active
ws.title = "Monthly Comparison"

# Write headers
headers = list(monthly_table.columns)
for col_idx, header in enumerate(headers, start=1):
    cell = ws.cell(row=1, column=col_idx, value=header)
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal='center', vertical='center')

# Write data rows
for row_idx, row_data in enumerate(monthly_table.values, start=2):
    for col_idx, value in enumerate(row_data, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=value)
        cell.alignment = Alignment(horizontal='center', vertical='center')
        
        # Apply color coding to MoM/MTD column (last column)
        if col_idx == len(headers):
            if isinstance(value, str) and value != '':
                if value.startswith('+'):
                    cell.font = Font(color="00B050", bold=True)  # Green
                elif value.startswith('-'):
                    cell.font = Font(color="FF0000", bold=True)  # Red

# Adjust column widths
ws.column_dimensions['A'].width = 35
ws.column_dimensions['B'].width = 25
ws.column_dimensions['C'].width = 25
ws.column_dimensions['D'].width = 15

wb.save(excel_output_file)

print(f"✓ Monthly comparison table with color coding saved to: {excel_output_file}")
print("\n" + "=" * 70)
print("MONTHLY TABLE GENERATION COMPLETE!")
print("=" * 70)

✓ Monthly comparison table with color coding saved to: ../outputs/sales_metrics_analysis/monthly_sales_metrics.xlsx

MONTHLY TABLE GENERATION COMPLETE!
